# S6E8 — modelling pipeline

The experiment notebook. `eda-s6e8.ipynb` is the frozen EDA record; this is where probes run.

Everything that varies between runs lives in one `CFG` dict, overridable from the `S6E8_CFG`
environment variable. On Kaggle the variable is absent and the defaults below are used, so the same
file is both the local probe runner and the pushed kernel.

**Strict-twin discipline** (`KAGGLE_PLAYBOOK.md` §3): a probe changes exactly one `CFG` field from
the champion. `scripts/run_local.py --diff-vs` checks this before the run starts.

In [ ]:
import json, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
T_START = time.time()

# ---- frozen contract (README.md). Never overridable by CFG. ----------------
SEED, N_FOLDS = 42, 5
TARGET, ID = "addicted_label", "id"
COMP = "playground-series-s6e8"

# ---- everything a probe is allowed to change ------------------------------
DEFAULTS = {
    "run_tag":       "champion",
    "learner":       "lgb",        # lgb | xgb | cat
    "model_seed":    42,           # model randomness ONLY -- never the CV split seed
    "fe_interaction": False,       # B1: opens_per_hour, notif_per_hour, notif_per_open
    "fe_composition": False,       # B2: other_hours, share_*, weekend_ratio, screen_total
    "fe_normalization": False,     # B3: free_time, screen_per_sleep, screen_per_age, social_per_age
    "drop_flat_cats": False,       # B5: drop gender/stress_level/academic_work_impact
    "lgb_params":    {},           # merged over the baseline params
    "n_estimators":  8000,
    "early_stop":    100,
}
CFG = {**DEFAULTS, **json.loads(os.environ.get("S6E8_CFG", "{}"))}
assert set(CFG) <= set(DEFAULTS), f"unknown CFG keys: {set(CFG) - set(DEFAULTS)}"

KAGGLE_DIR = Path("/kaggle/input/competitions") / COMP
ON_KAGGLE  = KAGGLE_DIR.exists()
DATA_DIR   = KAGGLE_DIR if ON_KAGGLE else Path("data")
OUT_DIR    = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("S6E8_OUT", "experiments/preds/local"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"on_kaggle={ON_KAGGLE}  out={OUT_DIR}")
print("CFG:", json.dumps(CFG, indent=1))

## Load and feature engineering

**Leakage note.** Every engineered feature below is *row-wise arithmetic on that row's own values* —
no aggregation across rows, no reference to `y`. It is therefore trivially leak-free and correctly
computed once, before the CV loop. This is the deliberate exception to `README.md`'s
fit-inside-the-fold rule: that rule exists for transforms that *learn* something from the data
(target encoders, bin edges, scalers), and none of these do. Do not "fix" this by moving it into the
fold loop — it would change nothing and cost 5× the compute.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test  = pd.read_csv(DATA_DIR / "test.csv")
sub   = pd.read_csv(DATA_DIR / "sample_submission.csv")

RAW_CAT = ["gender", "stress_level", "academic_work_impact"]
RAW_NUM = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
           "work_study_hours", "sleep_hours", "notifications_per_day",
           "app_opens_per_day", "weekend_screen_time"]

# Denominators must be strictly positive or the ratios below silently produce inf.
# Assert it rather than trusting the EDA snapshot.
for col, lo in [("daily_screen_time_hours", 0), ("app_opens_per_day", 0),
                ("sleep_hours", 0), ("age", 0)]:
    for name, df in (("train", train), ("test", test)):
        bad = (df[col].dropna() <= lo).sum()
        assert bad == 0, f"{name}.{col} has {bad} values <= {lo}; ratios would be inf"
print("denominator safety: all strictly positive in train and test")


def engineer(df):
    # Row-wise, unsupervised, leak-free by construction. Returns (df, added_cols).
    out, added = df.copy(), []

    if CFG["fe_interaction"]:
        # B1. The rank-gap finding: these two rank 9th/6th on marginal AUC but 1st/2nd on
        # split usage. A GBDT cannot express x/y with axis-aligned splits, so it burns
        # capacity approximating these; hand them over explicitly.
        out["opens_per_hour"] = df.app_opens_per_day / df.daily_screen_time_hours
        out["notif_per_hour"] = df.notifications_per_day / df.daily_screen_time_hours
        out["notif_per_open"] = df.notifications_per_day / df.app_opens_per_day
        added += ["opens_per_hour", "notif_per_hour", "notif_per_open"]

    if CFG["fe_composition"]:
        # B2. daily_screen_time is a confirmed composition: total - parts is NEVER negative
        # across 374,297 complete rows, so "other_hours" is a real latent variable.
        parts = df.social_media_hours + df.gaming_hours + df.work_study_hours
        out["other_hours"]   = df.daily_screen_time_hours - parts
        out["share_social"]  = df.social_media_hours / df.daily_screen_time_hours
        out["share_gaming"]  = df.gaming_hours / df.daily_screen_time_hours
        out["share_work"]    = df.work_study_hours / df.daily_screen_time_hours
        out["share_other"]   = out["other_hours"] / df.daily_screen_time_hours
        out["weekend_ratio"] = df.weekend_screen_time / df.daily_screen_time_hours
        out["screen_total"]  = df.daily_screen_time_hours + df.weekend_screen_time
        added += ["other_hours", "share_social", "share_gaming", "share_work",
                  "share_other", "weekend_ratio", "screen_total"]

    if CFG["fe_normalization"]:
        # B3. Absolute screen time means something different at 18 than at 35, and
        # against 5 vs 9 hours of sleep.
        out["free_time"]        = 24 - df.sleep_hours - df.daily_screen_time_hours
        out["screen_per_sleep"] = df.daily_screen_time_hours / df.sleep_hours
        out["screen_per_age"]   = df.daily_screen_time_hours / df.age
        out["social_per_age"]   = df.social_media_hours / df.age
        added += ["free_time", "screen_per_sleep", "screen_per_age", "social_per_age"]

    return out, added


train_fe, added = engineer(train)
test_fe,  _     = engineer(test)

cat_cols = [] if CFG["drop_flat_cats"] else RAW_CAT
FEATURES = RAW_NUM + added + cat_cols
print(f"{len(FEATURES)} features = {len(RAW_NUM)} raw num + {len(added)} engineered + {len(cat_cols)} cat")
print(f"engineered: {added or '(none)'}")

In [ ]:
# ---- FE correctness, asserted not assumed ---------------------------------
for name, df in (("train", train_fe), ("test", test_fe)):
    for c in added:
        v = df[c].to_numpy(dtype="float64")
        finite = v[~np.isnan(v)]
        assert np.isfinite(finite).all(), f"{name}.{c} contains inf"
if CFG["fe_composition"]:
    for name, df in (("train", train_fe), ("test", test_fe)):
        neg = (df["other_hours"].dropna() < -1e-9).sum()
        assert neg == 0, f"{name}.other_hours negative on {neg} rows -- composition assumption broken"
    print("composition check: other_hours >= 0 everywhere in train and test")

# Row-wise purity: recomputing FE on a shuffled subset must give identical values.
# This is what proves no cross-row aggregation crept in (which WOULD be a leak).
if added:
    probe = train.sample(2000, random_state=0)
    re_fe, _ = engineer(probe)
    ref = train_fe.loc[probe.index, added]
    assert np.allclose(re_fe[added].to_numpy(dtype="float64"),
                       ref.to_numpy(dtype="float64"), equal_nan=True), \
        "FE is not row-wise -- values changed when computed on a subset"
    print("row-wise purity: FE identical on a 2000-row shuffled subset -> no cross-row leakage")

print(f"\nengineered-column coverage (non-null %):")
for c in added:
    print(f"  {c:<20} {train_fe[c].notna().mean()*100:5.1f}%")

## Cross-validation

The frozen split, unchanged: `StratifiedKFold(5, shuffle=True, random_state=42)`. `CFG["model_seed"]`
moves the *model's* randomness only — the partition never moves, which is what keeps every archived
OOF matrix blendable with every other.

In [ ]:
X, X_test = train_fe[FEATURES].copy(), test_fe[FEATURES].copy()
for c in cat_cols:
    levels = pd.Index(sorted(set(train[c].dropna()) | set(test[c].dropna())))
    X[c]      = pd.Categorical(X[c], categories=levels)
    X_test[c] = pd.Categorical(X_test[c], categories=levels)
y = train[TARGET].values

BASE_PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=31,
                   n_jobs=-1, verbose=-1)
params = {**BASE_PARAMS, **CFG["lgb_params"],
          "n_estimators": CFG["n_estimators"], "random_state": CFG["model_seed"]}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
assert skf.n_splits == 5 and skf.shuffle is True and skf.random_state == 42, "CV split drifted!"

oof        = np.zeros(len(X))
fold_id    = np.full(len(X), -1, dtype=int)
test_proba = np.zeros(len(X_test))
fold_aucs, best_iters = [], []

for k, (itr, iva) in enumerate(skf.split(X, y)):
    fold_id[iva] = k
    model = lgb.LGBMClassifier(**params)
    model.fit(X.iloc[itr], y[itr], eval_set=[(X.iloc[iva], y[iva])], eval_metric="auc",
              callbacks=[lgb.early_stopping(CFG["early_stop"], verbose=False)])
    oof[iva] = model.predict_proba(X.iloc[iva])[:, 1]
    test_proba += model.predict_proba(X_test)[:, 1] / N_FOLDS
    fold_aucs.append(roc_auc_score(y[iva], oof[iva]))
    best_iters.append(int(model.best_iteration_))
    print(f"  fold {k}: AUC={fold_aucs[-1]:.6f}  best_iter={best_iters[-1]}", flush=True)

oof_auc = roc_auc_score(y, oof)
assert (fold_id >= 0).all()
assert max(best_iters) < CFG["n_estimators"], \
    f"a fold hit the n_estimators cap ({max(best_iters)}) -- under-trained, raise it"

print(f"\nOOF AUC : {oof_auc:.6f}")
print(f"folds   : {np.mean(fold_aucs):.6f} +/- {np.std(fold_aucs):.6f}  (spread {max(fold_aucs)-min(fold_aucs):.6f})")
print("fold spread is EVALUATION-FOLD DIFFICULTY, not test-prediction variance.")

In [ ]:
imp = (pd.Series(model.feature_importances_, index=FEATURES)
         / model.feature_importances_.sum() * 100).sort_values(ascending=False)
print("feature importance (last fold, % of splits):")
print(imp.round(2).to_string())

## Artifacts

In [ ]:
LEARNER = CFG["learner"]
pd.DataFrame({ID: train[ID], "fold": fold_id, "proba": oof}).to_csv(
    OUT_DIR / f"oof_proba_{LEARNER}.csv", index=False)
pd.DataFrame({ID: test[ID], "proba": test_proba}).to_csv(
    OUT_DIR / f"test_proba_{LEARNER}.csv", index=False)

submission = pd.DataFrame({ID: test[ID], TARGET: test_proba})
submission.to_csv(OUT_DIR / "submission.csv", index=False)

assert list(submission.columns) == [ID, TARGET]
assert len(submission) == len(sub)
assert (submission[ID].values == sub[ID].values).all(), "ids must match sample_submission in ORDER"
assert submission[TARGET].notna().all() and submission[TARGET].between(0, 1).all()
print(f"submission validated: {len(submission):,} rows, "
      f"mean {submission[TARGET].mean():.5f} (train base rate 0.70942)")

In [ ]:
RUN_METRICS = {
    "run_tag":            CFG["run_tag"],
    "final_oof_auc":      round(float(oof_auc), 6),
    f"{LEARNER}_oof_auc": round(float(oof_auc), 6),
    "fold_aucs":          [round(float(a), 6) for a in fold_aucs],
    "fold_auc_mean":      round(float(np.mean(fold_aucs)), 6),
    "fold_auc_std":       round(float(np.std(fold_aucs)), 6),
    "best_iters":         best_iters,
    "n_features":         len(FEATURES),
    "engineered":         added,
    "model_seed":         CFG["model_seed"],
    "n_folds":            N_FOLDS,
    "cv_seed":            SEED,
    "cfg":                CFG,
    "notebook_runtime_sec": round(time.time() - T_START, 1),
}
print("RUN_METRICS_JSON:" + json.dumps(RUN_METRICS))